# LLM interactions

Install packages

In [1]:
#%pip install pandas                # Python package to work with data
#%pip install numpy                  # Python package for computing
#%pip install emoji                 # Emoji library

#%pip install nltk                  # Natural Language Toolkit
#import nltk
#nltk.download()
#https://www.nltk.org/install.html

Load dataset

In [ ]:
import pandas as pd
df = pd.read_csv("LLM_file.csv", sep = ';')
display(df)

# Display datatypes
display(df.dtypes)

,id,LLM,# of task,# of query,query,LLM response,LLM response - plain,Clarified own role,Defined role for LLM,Pictures/graphs shown in results,LLM provided code,Participant aborted search
0,11,2,1,1,data to train llms with,Large Language Models (LLMs) require vast amou...,Large Language Models (LLMs) require vast amou...,N,N,NaN,NaN,NaN
1,11,2,1,2,do you have actual datasets?,I apologize for the confusion in my previous r...,I apologize for the confusion in my previous r...,N,N,NaN,NaN,NaN
2,11,2,1,3,Where can I find open-source datasets for LLM ...,There are several open-source datasets availab...,There are several open-source datasets availab...,N,N,NaN,NaN,NaN
3,11,2,1,4,give me a link to,"I apologize, but I cannot provide a direct lin...","I apologize, but I cannot provide a direct lin...",N,N,NaN,NaN,NaN
4,11,2,1,5,give me a link to The Pile,"I apologize, but I cannot provide direct links...","I apologize, but I cannot provide direct links...",N,N,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
291,44,2,2,6,How did the Ordonnance de la Marine influence ...,"The Ordonnance de la Marine, issued by Louis X...","The Ordonnance de la Marine, issued by Louis X...",N,NaN,NaN,NaN,NaN
292,45,2,1,1,find me datasets that can answer the question ...,Several datasets from recent studies can help ...,Several datasets from recent studies can help ...,N,N,NaN,NaN,NaN
293,45,2,2,prompt,"For the following chat, I want you to act as t...","Certainly! I'll be happy to act as Leslie, you...","Certainly! I'll be happy to act as Leslie, you...",NaN,NaN,NaN,NaN,NaN
294,45,2,2,1,You know that I'm always wondering why it is m...,Hey there! I totally get why you're curious ab...,Hey there! I totally get why you're curious ab...,N,NaN,NaN,NaN,NaN


id                                   int64
LLM                                  int64
# of task                            int64
# of query                          object
query                               object
LLM response                        object
LLM response - plain                object
Clarified own role                  object
Defined role for LLM                object
Pictures/graphs shown in results    object
LLM provided code                   object
Participant aborted search          object
dtype: object

## Word counts

In [3]:
df['query word count'] = df['query'].str.split().str.len()

# Delete citations in Perplexity (were not displayed on screen and are not counted in response length)
df['LLM response temp'] = df['LLM response - plain'].str.split('Citations:', n=-1).str.get(0)
df['LLM response word count'] = df['LLM response temp'].str.split().str.len()
df = df.drop(['LLM response temp'], axis = 1)

#display(df)

## Humanisation

### Does LLM response contain Emoji(s)?

In [4]:
import emoji

df['emoji in query'] = (df['query']).apply(emoji.emoji_count)
df['emoji in LLM response'] = (df['LLM response']).apply(emoji.emoji_count)
#display(df.head(30))

### Does the query mention "Leslie"?

In [5]:
leslie = ['Leslie', 'leslie']
df['Leslie in query'] = df.loc[:, ('query')].str.contains('|'.join(leslie))
#display(df)


## Keyword search vs. conversation



With part of speech tagging (NLTK): Count the number of nouns (likely these are keywords) and build ratio to the total number of words in one query.

A keywords search has more nouns compared to total words, f.ex. "armateur in the French slave trade". A conversation has further "filling words", like greetings, verbs, pronouns, etc. Hence, a lower noun to total words ratio.

Sources: 
- https://www.geeksforgeeks.org/nlp-part-of-speech-default-tagging/ (wrong command for pos_tag)
- https://www.nltk.org/api/nltk.tag.pos_tag.html

Rule based Tags:

12. 	NN 	Noun, singular or mass
13. 	NNS 	Noun, plural
14. 	NNP 	Proper noun, singular
15. 	NNPS 	Proper noun, plural 
(from https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html)

In [6]:
# Importing the NLTK library
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
from functools import reduce

df['QUERY_WORD_TOKENS'] = df['query'].str.lower()
df['QUERY_WORD_TOKENS'] = (df['QUERY_WORD_TOKENS']).apply(word_tokenize)
df['RESPONSE_WORD_TOKENS'] = (df['LLM response']).str.lower()
df['RESPONSE_WORD_TOKENS'] = (df['RESPONSE_WORD_TOKENS']).apply(word_tokenize)

# Performing PoS tagging
df['QUERY_WORD_TAGS'] = df['QUERY_WORD_TOKENS'].apply(pos_tag)

# Convert list of tuples into list
unpack_tag = lambda term_tag: term_tag[1]
filter_tags = lambda l: list(map(unpack_tag, l))
df['QUERY_WORD_TAGS'] = df['QUERY_WORD_TAGS'].apply(filter_tags)

# Count all nouns in a query
noun = ['NN', 'NNP', 'NNS', 'NNPS']

def countnouns(xs):
    for x in xs:
        if x in noun:
            return len([x for x in xs if x in noun])
    return 0

df['query noun count'] = df['QUERY_WORD_TAGS'].apply(countnouns)

# Calculate ratio
df['query noun ratio'] = (df['query noun count'] / df['query word count'])

#display(df)

## Drop analysis variables

In [7]:
# Drop token and tag columns
df = df.drop(['QUERY_WORD_TOKENS', 'RESPONSE_WORD_TOKENS', 'QUERY_WORD_TAGS'], axis = 1)

display(df)

,id,LLM,# of task,# of query,query,LLM response,LLM response - plain,Clarified own role,Defined role for LLM,Pictures/graphs shown in results,LLM provided code,Participant aborted search,query word count,LLM response word count,emoji in query,emoji in LLM response,Leslie in query,query noun count,query noun ratio
0,11,2,1,1,data to train llms with,Large Language Models (LLMs) require vast amou...,Large Language Models (LLMs) require vast amou...,N,N,NaN,NaN,NaN,5,254,0,0,False,2,0.400000
1,11,2,1,2,do you have actual datasets?,I apologize for the confusion in my previous r...,I apologize for the confusion in my previous r...,N,N,NaN,NaN,NaN,5,193,0,0,False,1,0.200000
2,11,2,1,3,Where can I find open-source datasets for LLM ...,There are several open-source datasets availab...,There are several open-source datasets availab...,N,N,NaN,NaN,NaN,9,275,0,0,False,2,0.222222
3,11,2,1,4,give me a link to,"I apologize, but I cannot provide a direct lin...","I apologize, but I cannot provide a direct lin...",N,N,NaN,NaN,NaN,5,174,0,0,False,1,0.200000
4,11,2,1,5,give me a link to The Pile,"I apologize, but I cannot provide direct links...","I apologize, but I cannot provide direct links...",N,N,NaN,NaN,NaN,7,130,0,0,False,2,0.285714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,44,2,2,6,How did the Ordonnance de la Marine influence ...,"The Ordonnance de la Marine, issued by Louis X...","The Ordonnance de la Marine, issued by Louis X...",N,NaN,NaN,NaN,NaN,13,297,0,0,False,5,0.384615
292,45,2,1,1,find me datasets that can answer the question ...,Several datasets from recent studies can help ...,Several datasets from recent studies can help ...,N,N,NaN,NaN,NaN,18,274,0,0,False,9,0.500000
293,45,2,2,prompt,"For the following chat, I want you to act as t...","Certainly! I'll be happy to act as Leslie, you...","Certainly! I'll be happy to act as Leslie, you...",NaN,NaN,NaN,NaN,NaN,63,52,0,0,True,15,0.238095
294,45,2,2,1,You know that I'm always wondering why it is m...,Hey there! I totally get why you're curious ab...,Hey there! I totally get why you're curious ab...,N,NaN,NaN,NaN,NaN,37,334,0,0,False,7,0.189189


## Save to csv

In [8]:
# Display datatypes
display(df.dtypes)

# Save to csv
df.to_csv("LLM_file_WithInfoCoded.csv", sep = ';')

id                                    int64
LLM                                   int64
# of task                             int64
# of query                           object
query                                object
LLM response                         object
LLM response - plain                 object
Clarified own role                   object
Defined role for LLM                 object
Pictures/graphs shown in results     object
LLM provided code                    object
Participant aborted search           object
query word count                      int64
LLM response word count               int64
emoji in query                        int64
emoji in LLM response                 int64
Leslie in query                        bool
query noun count                      int64
query noun ratio                    float64
dtype: object

## Create dataset on individual level (one row per participant) 

Create variables from df: https://stackoverflow.com/questions/33445009/pandas-new-column-from-groupby-averages

In [9]:
import numpy as np

# Averages
df['Average query length'] = df.groupby(['id', '# of task'])['query word count'].transform('mean')
df['Average response length'] = df.groupby(['id', '# of task'])['LLM response word count'].transform('mean')
df['Average query noun count'] = df.groupby(['id', '# of task'])['query noun count'].transform('mean')
df['Average query noun ratio'] = df.groupby(['id', '# of task'])['query noun ratio'].transform('mean')
df['Average emojis in LLM response'] = df.groupby(['id', '# of task'])['emoji in LLM response'].transform('mean')
display(df)
display(df.dtypes)

# Create flattened dataset - Two rows per participant for task 1 and task 2
df_flatted = df.groupby(['id', '# of task']).last()
# Convert number of queries to numeric
df_flatted['Total number of queries'] = df_flatted['# of query'].astype('int64')
# Keep columns
df_flatted = df_flatted[['LLM', 'Total number of queries', 'Clarified own role', 'Defined role for LLM', 'Average query length', 'Average response length', 'Average query noun count', 'Average query noun ratio', 'Average emojis in LLM response']]

# Create flat dataset - One row per participant
df_flatted = df_flatted.reset_index()
df_flat = df_flatted.pivot(index='id', columns='# of task')
df_flat.columns = df_flat.columns.map(lambda x: f'{x[1]}_{x[0]}')

# Drop and rename columns
df_flat = df_flat.drop(['2_LLM', '2_Defined role for LLM'], axis = 1)
df_flat = df_flat.rename(columns={'1_LLM': 'LLM', \
    '1_Total number of queries': 'Total number of queries - Task 1', '2_Total number of queries': 'Total number of queries - Task 2', \
    '1_Clarified own role': 'Clarified own role - Task 1', '2_Clarified own role': 'Clarified own role - Task 2', \
    '1_Defined role for LLM': 'Defined role for LLM - Task 1', \
    '1_Average query length': 'Average query length - Task 1', '2_Average query length': 'Average query length - Task 2', \
    '1_Average response length': 'Average response length - Task 1', '2_Average response length': 'Average response length - Task 2', \
    '1_Average query noun count': 'Average query noun count - Task 1', '2_Average query noun count': 'Average query noun count - Task 2', \
    '1_Average query noun ratio': 'Average query noun ratio - Task 1', '2_Average query noun ratio': 'Average query noun ratio - Task 2', \
    '1_Average emojis in LLM response': 'Average emojis in LLM response - Task 1', '2_Average emojis in LLM response': 'Average emojis in LLM response - Task 2', \
    })
display(df_flat)
display(df_flat.dtypes)


,id,LLM,# of task,# of query,query,LLM response,LLM response - plain,Clarified own role,Defined role for LLM,Pictures/graphs shown in results,...,emoji in query,emoji in LLM response,Leslie in query,query noun count,query noun ratio,Average query length,Average response length,Average query noun count,Average query noun ratio,Average emojis in LLM response
0,11,2,1,1,data to train llms with,Large Language Models (LLMs) require vast amou...,Large Language Models (LLMs) require vast amou...,N,N,NaN,...,0,0,False,2,0.400000,6.200000,205.200000,1.600000,0.261587,0.0
1,11,2,1,2,do you have actual datasets?,I apologize for the confusion in my previous r...,I apologize for the confusion in my previous r...,N,N,NaN,...,0,0,False,1,0.200000,6.200000,205.200000,1.600000,0.261587,0.0
2,11,2,1,3,Where can I find open-source datasets for LLM ...,There are several open-source datasets availab...,There are several open-source datasets availab...,N,N,NaN,...,0,0,False,2,0.222222,6.200000,205.200000,1.600000,0.261587,0.0
3,11,2,1,4,give me a link to,"I apologize, but I cannot provide a direct lin...","I apologize, but I cannot provide a direct lin...",N,N,NaN,...,0,0,False,1,0.200000,6.200000,205.200000,1.600000,0.261587,0.0
4,11,2,1,5,give me a link to The Pile,"I apologize, but I cannot provide direct links...","I apologize, but I cannot provide direct links...",N,N,NaN,...,0,0,False,2,0.285714,6.200000,205.200000,1.600000,0.261587,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,44,2,2,6,How did the Ordonnance de la Marine influence ...,"The Ordonnance de la Marine, issued by Louis X...","The Ordonnance de la Marine, issued by Louis X...",N,NaN,NaN,...,0,0,False,5,0.384615,18.285714,202.714286,5.428571,0.337627,0.0
292,45,2,1,1,find me datasets that can answer the question ...,Several datasets from recent studies can help ...,Several datasets from recent studies can help ...,N,N,NaN,...,0,0,False,9,0.500000,18.000000,274.000000,9.000000,0.500000,0.0
293,45,2,2,prompt,"For the following chat, I want you to act as t...","Certainly! I'll be happy to act as Leslie, you...","Certainly! I'll be happy to act as Leslie, you...",NaN,NaN,NaN,...,0,0,True,15,0.238095,38.333333,197.333333,9.000000,0.253539,0.0
294,45,2,2,1,You know that I'm always wondering why it is m...,Hey there! I totally get why you're curious ab...,Hey there! I totally get why you're curious ab...,N,NaN,NaN,...,0,0,False,7,0.189189,38.333333,197.333333,9.000000,0.253539,0.0


id                                    int64
LLM                                   int64
# of task                             int64
# of query                           object
query                                object
LLM response                         object
LLM response - plain                 object
Clarified own role                   object
Defined role for LLM                 object
Pictures/graphs shown in results     object
LLM provided code                    object
Participant aborted search           object
query word count                      int64
LLM response word count               int64
emoji in query                        int64
emoji in LLM response                 int64
Leslie in query                        bool
query noun count                      int64
query noun ratio                    float64
Average query length                float64
Average response length             float64
Average query noun count            float64
Average query noun ratio        

,LLM,Total number of queries - Task 1,Total number of queries - Task 2,Clarified own role - Task 1,Clarified own role - Task 2,Defined role for LLM - Task 1,Average query length - Task 1,Average query length - Task 2,Average response length - Task 1,Average response length - Task 2,Average query noun count - Task 1,Average query noun count - Task 2,Average query noun ratio - Task 1,Average query noun ratio - Task 2,Average emojis in LLM response - Task 1,Average emojis in LLM response - Task 2
id,,,,,,,,,,,,,,,,
11,2,5,3,N,N,N,6.200000,46.250000,205.200000,186.750000,1.600000,13.500000,0.261587,0.347558,0.000000,0.000000
12,1,3,3,N,N,N,19.000000,39.500000,367.333333,296.500000,7.000000,9.500000,0.366374,0.183956,1.000000,12.500000
13,1,5,2,N,Y,N,19.400000,45.333333,365.600000,222.666667,8.000000,13.666667,0.446300,0.303860,2.400000,0.666667
14,1,4,5,N,N,N,16.000000,19.666667,405.750000,199.666667,5.500000,5.833333,0.338588,0.452907,23.500000,3.333333
15,1,7,9,N,N,N,9.285714,15.300000,122.142857,361.700000,2.857143,3.700000,0.298629,0.239008,1.285714,3.400000
16,2,2,2,Y,N,N,28.000000,35.333333,130.500000,115.000000,13.500000,9.666667,0.480952,0.298003,0.000000,0.000000
17,2,11,11,N,N,N,8.818182,15.583333,166.090909,155.416667,3.000000,4.083333,0.299082,0.277464,0.000000,0.000000
19,1,5,7,N,N,N,9.400000,15.625000,225.000000,144.875000,3.000000,4.500000,0.322078,0.320483,0.600000,2.375000
20,2,3,4,Y,N,N,21.000000,25.000000,360.666667,232.000000,7.666667,6.000000,0.595960,0.305801,0.000000,0.000000


LLM                                          int64
Total number of queries - Task 1             int64
Total number of queries - Task 2             int64
Clarified own role - Task 1                 object
Clarified own role - Task 2                 object
Defined role for LLM - Task 1               object
Average query length - Task 1              float64
Average query length - Task 2              float64
Average response length - Task 1           float64
Average response length - Task 2           float64
Average query noun count - Task 1          float64
Average query noun count - Task 2          float64
Average query noun ratio - Task 1          float64
Average query noun ratio - Task 2          float64
Average emojis in LLM response - Task 1    float64
Average emojis in LLM response - Task 2    float64
dtype: object

In [10]:
df_flat.to_csv("ParticipantsDataFromLLM.csv", sep = ';')